# 02b. Tracking cascade mechanism

**Purpose**: check that the tracking cascade ran the way it was designed to before asking whether
it worked. `02a` scores a cascade run against its reference like any other run; this notebook
opens the mechanism up, because a null result only informs the reader if the implementation can
be ruled out as its cause.

**Input**: the same cached `per_image_results.csv` files `02a` uses. A cascade run carries extra
per-frame columns recording which tier fired, how much magnification the region actually bought,
how much of the tool the region box covered, and how often the tracker lost lock.

**Output**: `outputs/cascade-mechanism/`. These artefacts cover both cascade runs at once, so
they belong to no single pair and are not written into the `02a` comparison directories.

The cascade only crops when the bootstrap detection leaves magnification on the table. Otherwise
it falls back to a whole-frame pass which is, by construction, the reference's own code path, so
those frames should differ from the reference by exactly zero. They do, and that is the check
that confirms a run is scoring the checkpoints its configuration names.


## 1. Install dependencies
Run this cell to install the packages required for this notebook, pinned in `requirements.txt`.


In [ ]:
from pathlib import Path

_req = Path("requirements.txt")
if not _req.exists():
    _req = Path("02_results_comparison") / "requirements.txt"
if not _req.exists():
    raise FileNotFoundError(f"requirements.txt not found from cwd={Path.cwd()}")
get_ipython().run_line_magic("pip", f"install -q -r {_req.as_posix()}")

import matplotlib, numpy, pandas, scipy
print(f"pandas {pandas.__version__} | numpy {numpy.__version__} "
      f"| matplotlib {matplotlib.__version__} | scipy {scipy.__version__}")


## 2. Setup

Two cascade runs exist, one on the baseline's weights and one on the ROI-crop fine-tuned weights.
Each is paired with the run whose checkpoints it actually carries, read out of its own
`results.json` rather than assumed, since the run names do not always say.


In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path.cwd()
if not (_NB_DIR / "compare_common.py").exists():
    _NB_DIR = _NB_DIR / "02_results_comparison"
if not (_NB_DIR / "compare_common.py").exists():
    raise ModuleNotFoundError(f"compare_common.py not found from cwd={Path.cwd()}")
sys.path.insert(0, str(_NB_DIR.resolve()))

import importlib
import compare_common
importlib.reload(compare_common)  # pick up edits to compare_common without a kernel restart
from compare_common import *
from IPython.display import Image, display

CASCADE_ARMS = ["cascade-redetect-clean", "cascade-roift-neg-redetect"]

OUT = OUTPUT_ROOT / "cascade-mechanism"
TABLES, FIGURES = OUT / "tables", OUT / "figures"

loaded = {}
for arm in CASCADE_ARMS:
    reference = declared_weights_source(results(arm))
    frames, gt = load_runs([reference, arm])
    loaded[arm] = dict(reference=reference, frames=frames, gt=gt, strata=build_strata(gt))
    print(f"{arm:<30s} scores the weights of {reference}")
print(f"output    : {OUT}")


## 3. Did it fire, and where

Tier shares first. `roi` is the intended path; the `bootstrap_*` tiers are frames where the first
full-frame pass found nothing to track or found something already filling the frame, and
`fullframe_after_empty_roi` is the safety net for a region that came back empty.

The column that matters is the IoU change per tier. Every non-`roi` tier is the reference's own
whole-frame path, so a non-zero number there would mean the run differs from its reference for
some reason other than the cropping, and no conclusion about the cropping would follow.


In [ ]:
for arm, ctx in loaded.items():
    diag = cascade_diagnostics(ctx["frames"], ctx["strata"], arm, ctx["reference"])
    print(f"{RUNS[arm].label}, against {RUNS[ctx['reference']].label}")
    display(save_table(diag["by_tier"], f"cascade_tiers_{arm}", TABLES))


## 4. How much magnification it actually bought

The intervention's whole premise is that a fixed 512 x 512 input budget spent on the tool
resolves more of it than the same budget spent on the whole frame. That only holds if the crop is
meaningfully smaller than the frame.

`cascade_roi_zoom` is the magnification the region achieved. Its distribution is the honest
answer to whether the mechanism had anything to work with, and it is measured rather than assumed
because the region is fitted to the model's own detection, which on a large tool leaves almost no
room.


In [ ]:
fig, axes = plt.subplots(1, len(loaded), figsize=(5.6 * len(loaded), 3.8), squeeze=False)
for ax, (arm, ctx) in zip(axes[0], loaded.items()):
    zoom = pd.to_numeric(ctx["frames"][arm]["cascade_roi_zoom"], errors="coerce").dropna()
    ax.hist(zoom, bins=50, range=(1.0, 2.6), color=PRIMARY)
    ax.axvline(float(zoom.median()), color=RULE, lw=1, ls="--")
    ax.set_title(f"{label(arm)}\nmedian {zoom.median():.4f}, "
                 f"{100 * (zoom <= 1.0).mean():.1f}% no gain", fontsize=9)
    ax.set_xlabel("achieved magnification")
    ax.set_ylabel("frames")
    ax.grid(alpha=0.3)
fig.tight_layout()
display(Image(filename=str(save_fig(fig, "cascade_zoom_distribution", FIGURES))))


## 5. What magnification costs

Binning the effect by achieved zoom separates the composite's union rule from the scale shift it
introduces. The bin where no crop applies is the control: those frames go through the reference's
own path, so any trend across the remaining bins is the magnification and nothing else.

Precision falling monotonically with zoom while IoU does not is the signature of a model asked to
segment at a scale it was not trained on, and it is what motivated fine-tuning on crops rather
than only cropping at test time.


In [ ]:
for arm, ctx in loaded.items():
    diag = cascade_diagnostics(ctx["frames"], ctx["strata"], arm, ctx["reference"])
    print(RUNS[arm].label)
    display(save_table(diag["by_zoom"], f"cascade_zoom_{arm}", TABLES))


## 6. Truncation, and the coverage check

The first cascade attempt pasted the region prediction onto a blank canvas, so every tool pixel
outside the box became a false negative that no threshold could recover. The composite in these
runs starts from the full-frame pass and lets each region act only inside its own box, which
makes the reference the worst case rather than the best.

`cascade_roi_cover_frac` records the share of the ground-truth tool the region boxes covered, so
the geometry that caused that failure can still be measured. Frames below 1.0 are ones where a
box did clip the tool; what matters is that they are no longer scored as if the missing part had
been predicted empty.


In [ ]:
rows = []
for arm, ctx in loaded.items():
    frames, reference = ctx["frames"], ctx["reference"]
    tool = np.asarray(ctx["strata"]["all tool frames"], dtype=bool)
    cover = pd.to_numeric(frames[arm]["cascade_roi_cover_frac"], errors="coerce")
    delta = frames[arm]["iou"].to_numpy(float) - frames[reference]["iou"].to_numpy(float)
    scored = cover.notna().to_numpy() & tool
    partial = scored & (cover.to_numpy() < 1.0)
    rows.append({"run": label(arm),
                 "frames with a region": int(scored.sum()),
                 "partial coverage": int(partial.sum()),
                 "partial coverage pct": 100.0 * partial.sum() / scored.sum(),
                 "mean coverage": float(cover[scored].mean()),
                 "delta iou, full coverage": float(np.nanmean(delta[scored & ~partial])),
                 "delta iou, partial coverage": float(np.nanmean(delta[partial]))})
display(save_table(pd.DataFrame(rows), "cascade_coverage", TABLES))


## 7. Recovery, and what the tracker cost

Three-tiered recovery: a template match against the cached crop handles a tool that briefly
vanished and returned, a confident match re-initialises the tracker without paying for a full
frame pass, and otherwise a full-frame redetection re-seeds the box. Recovery fires on an empty
region prediction as well as on reported lock loss, because a tracker was observed returning a
confident box after the target had left the frame entirely.

The counts below say how often the five-second horizon actually needed rescuing.


In [ ]:
rows = []
for arm, ctx in loaded.items():
    df = ctx["frames"][arm]
    num = lambda c: pd.to_numeric(df[c], errors="coerce")
    rows.append({"run": label(arm),
                 "lead-in seconds, median": float(num("cascade_lead_s").median()),
                 "tracked frames, median": float(num("cascade_n_track_frames").median()),
                 "frames with a lock loss pct": 100.0 * float((num("cascade_track_losses") > 0).mean()),
                 "template re-inits pct": 100.0 * float((num("cascade_tier2_reinits") > 0).mean()),
                 "full redetections pct": 100.0 * float((num("cascade_tier3_redetects") > 0).mean()),
                 "zoom capped pct": 100.0 * float((num("cascade_zoom_capped") > 0).mean()),
                 "regions per frame, mean": float(num("cascade_n_rois").mean())})
display(save_table(pd.DataFrame(rows), "cascade_recovery", TABLES))


## 8. What it cost on empty scenes

The cascade combines a region prediction into the full-frame one with a maximum, so it can only
add foreground. On a frame with no instrument at all there is nothing to add that is not an
error, and this is where that shows up.


In [ ]:
for arm, ctx in loaded.items():
    display(save_table(tool_free(ctx["frames"], ctx["strata"], ctx["reference"], [arm]),
                       f"tool_free_{arm}", TABLES))


## 9. Verdict

The mechanism questions and what the tables above answer:

1. **Did the region path fire?** Tier shares in section 3.
2. **Did it buy magnification?** The zoom distribution in section 4.
3. **Is the fallback the reference's own path?** The per-tier IoU change in section 3, which
   should be exactly zero outside the `roi` tier.
4. **Does magnification cost precision?** The zoom bins in section 5, read against the no-crop
   control bin.
5. **Did the truncation failure recur?** The coverage table in section 6.
6. **What did it cost on empty scenes?** Section 8.

A cascade that fires, magnifies, keeps its fallback clean and still does not move the
precision-recall curve is a negative result about the intervention. One that never fired, or
never magnified, would be a negative result about the implementation, and those are not the same
finding. Run `02a` on each cascade run for the outcome measures themselves.
